# Metinden hedef splat · MODEL SETTINGS

Tek nesne istemi → seçilen görüntü modeli → arka plan temizliği → **standart 3DGS target.ply**.
L4 24 GB, A100/H100 80 GB veya RTX PRO 6000 96 GB için ön ayarlardan başlayın; L4/A100/H100 kararlı TRELLIS, RTX PRO 6000 Max detay mesh yolunu kullanır.
TRELLIS.2 deneysel yüksek detay yoludur; Colab uçtan uca doğrulaması henüz yapılmadı. RTX PRO 6000/Blackwell, kararlı TRELLIS'in sabit cu121 paketleriyle uyumlu değildir: kararlı yol için uyumlu bir GPU oturumu seçin.
Model ağırlıkları yalnız Colab'da indirilir; yerel 8 GB GPU/T4 desteklenmez.
Görüntü ve geometri ayrı süreç/ortam kullanır; modeller aynı anda GPU'da tutulmaz.

**Uygulanan:** SDXL, FLUX.1-dev/schnell, FLUX.2-klein-4B, Qwen-Image; U2Net/BiRefNet;
TRELLIS Gaussian ve deneysel TRELLIS.2 mesh → görünümler → gsplat 1.5.3 fitting.
**Sonra:** Hunyuan3D-2.1 (devre dışı; özel lisans ve doku/CUDA bağımlılıkları henüz entegre edilmedi).
TRELLIS.2 kodu uygulanmıştır; temiz Colab GPU üzerinde uçtan uca doğrulama henüz yapılmadı.
VRAM eşikleri temkinli ön kontrollerdir; OOM olmaması veya daha iyi kalite garantisi değildir.


## 1. Model ayarları ve kalite

| Görüntü modeli | Kalite / hız / tarif VRAM eşiği | Lisans ve erişim |
|---|---|---|
| [SDXL base 1.0](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0) | Varsayılan; hızlı/ekonomik, 12 GiB görüntü, TRELLIS ile eşik 20 GiB | CreativeML Open RAIL++-M; açık erişim |
| [FLUX.1-dev · 12B](https://huggingface.co/black-forest-labs/FLUX.1-dev) | Ayrıntı/istem uyumu; 28 adım, daha yavaş, 36 GiB | Ticari olmayan model lisansı; HF lisans kabulü + HF_TOKEN |
| [FLUX.1-schnell · 12B](https://huggingface.co/black-forest-labs/FLUX.1-schnell) | 1–4 adımla hızlı; daha az ayrıntı kontrolü, 36 GiB | Apache 2.0; HF erişim kabulü + HF_TOKEN (gated) |
| [FLUX.2-klein · 4B](https://huggingface.co/black-forest-labs/FLUX.2-klein-4B) | 4 adımlı küçük/hızlı seçenek, 16 GiB | Apache 2.0, açık erişim |
| [Qwen-Image · 20B](https://huggingface.co/Qwen/Qwen-Image) | Ayrıntı/yazı güçlü; daha yavaş, 60 GiB | Apache 2.0, açık erişim |

Görüntü ortamı Python 3.11 / Torch 2.7.1 cu128 / Diffusers 0.37.0; CPU offload kullanır.
Boş sistem RAM eşiği sırasıyla 12 / 40 / 40 / 20 / 56 GiB; yüksek RAM Colab gerekebilir.
Negatif istem SDXL, FLUX.1-dev ve Qwen'de desteklenir. Schnell/klein'de kapalıdır; alan boş olmalıdır.
FLUX dev negatif istem kullanırken ek true CFG hesaplaması nedeniyle yavaşlar.

| Arka plan / 3D yolu | Açıklama / ihtiyaç | Lisans / durum |
|---|---|---|
| [U2Net](https://github.com/xuebinqin/U-2-Net) | CPU'da hızlı; ince kenar kaybı olabilir | Kaynak Apache 2.0, rembg MIT; ONNX ağırlığının ayrı lisansı doğrulanamadı |
| [BiRefNet](https://huggingface.co/ZhengPeng7/BiRefNet) | GPU'da daha ağır; saç/ince kenarlarda daha ayrıntılı maske | Resmi model kartı MIT, açık erişim; kod/ağırlık aynı SHA'ya sabit |
| [TRELLIS-image-large](https://huggingface.co/microsoft/TRELLIS-image-large) | Doğrudan Gaussian; hızlı/az bağımlılık, 20 GiB | MIT + DINOv2 Apache 2.0; cu121 ortamı Blackwell desteklemez |
| [TRELLIS.2-4B](https://huggingface.co/microsoft/TRELLIS.2-4B) | UV dokulu mesh → 512–2048px küresel görüş → adaptif gsplat; pipeline eşikleri 28/40/60 GiB | MIT; [DINOv3](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m) için **Meta manuel erişim onayı**, özel DINOv3 lisansı + HF_TOKEN |
| [Hunyuan3D-2.1](https://huggingface.co/tencent/Hunyuan3D-2.1) | Deneysel/sonra; bu tarifte çalışmaz | Tencent özel topluluk lisansı; geometri/doku CUDA kurulumu entegre edilmedi |

HF_TOKEN yalnız Colab Secrets veya ortam değişkeninden okunur; ayar hücresine token yazmayın.
TRELLIS.2'nin varsayılan BRIA modeli yüklenmez; seçtiğiniz U2Net/BiRefNet maskesi kullanılır.
FLUX.2-klein-9B/dev-32B varlığı doğrulandı; bu sürümde uygulanmadı, menüde yok.

GPU_PRESET API/UI'dan seçilince makul değerler doldurulur. Notebook içinde GPU_PRESET değiştirirseniz
APPLY_GPU_PRESET kutusunu işaretleyin: model/kalite alanları o ön ayarla değiştirilir.
Özel ayar yapmak için kutuyu kapatın. Formdaki açık sayısal değerler QUALITY_PRESET'ten önceliklidir.
Ultra geometri VRAM eşiğini en az 38 GiB yapar. SEED görüntü; TRELLIS_SEED geometri/fitting içindir.
Farklı GPU'larda bit düzeyinde eşitlik garanti edilmez. Ayar değiştirince yeni OUTPUT_DIR seçin.


In [ ]:
#@title MODEL SETTINGS · Modeller ve kalite
#@markdown GPU: L4 SDXL/U2Net; A100 FLUX dev/BiRefNet; H100/RTX PRO 6000 Qwen/BiRefNet. L4/A100/H100 kararlı TRELLIS; RTX PRO 6000 Max detay mesh varsayılanıdır. Blackwell kararlı cu121 yoluyla uyumsuzdur. Ön ayarı uygulamak için kutuyu açın.
GPU_PRESET = 'l4' #@param ["l4", "a100", "h100", "rtx_pro_6000"]
APPLY_GPU_PRESET = False #@param {"type":"boolean"}
#@markdown İngilizce tek nesne istemi, STYLE görünüm. Negatif istem SDXL/FLUX dev/Qwen için; schnell/klein'de boş olmalı (doğrulama engeller).
PROMPT = 'a red sports car' #@param {"type":"string"}
NEGATIVE_PROMPT = '' #@param {"type":"string"}
STYLE = '' #@param {"type":"string"}
SEED = 42 #@param {"type":"integer"}
QUALITY_PRESET = 'baseline' #@param ["baseline", "quality", "ultra", "max_detail"]
#@markdown SDXL12 GiB/Rail++; FLUX dev12B/36 GiB/ticari olmayan; schnell12B/36 GiB/Apache2/gated; klein4B/16 GiB/Apache2; Qwen20B/60 GiB/Apache2. FLUX.1 için lisans kabulü + HF_TOKEN.
IMAGE_MODEL = 'sdxl' #@param ["sdxl", "flux1_dev", "flux1_schnell", "flux2_klein_4b", "qwen_image"]
#@markdown U2Net CPU hızlı/basit kenar (ONNX lisansı doğrulanamadı); BiRefNet MIT, GPU'da daha ağır/ince kenarlarda iyi.
BACKGROUND_MODEL = 'u2net' #@param ["u2net", "birefnet"]
#@markdown TRELLIS: kararlı varsayılan, 20 GiB/MIT/doğrudan Gaussian. TRELLIS.2: deneysel mesh fitting, pipeline türüne göre 28/40/60 GiB/MIT; nvcc/g++ derlemeleri ve DINOv3 Meta manuel onayı+HF_TOKEN gerekir. Hunyuan3D deneysel/sonra, seçilemez.
RECONSTRUCTION_MODEL = 'trellis' #@param ["trellis", "trellis2"]
#@markdown Adım: artınca süre/ayrıntı artabilir (1–100; schnell en çok4). Guidance isteme bağlılığı artırır, aşırısı bozabilir; schnell0, klein1, Qwen true CFG.
IMAGE_STEPS = 25 #@param {"type":"integer"}
IMAGE_GUIDANCE = 7.0 #@param {"type":"number"}
#@markdown Çözünürlük: 512 hızlı; 1024 daha ayrıntılı/fazla VRAM. Yalnız görüntü içindir; mesh pipeline ve render çözünürlüğü ayrıca seçilir.
IMAGE_RESOLUTION = 1024 #@param [512, 768, 1024] {"type":"raw"}
#@markdown Geometri seed karşılaştırma içindir (0–2147483647). Sparse ana şekli, SLAT yüzey ayrıntısını işler; 1–100 adım/CFG0–20. Çok adım daha yavaş, yüksek CFG daha güçlü koşullandırma/bozulma riski.
TRELLIS_SEED = 42 #@param {"type":"integer"}
SPARSE_STEPS = 12 #@param {"type":"integer"}
SPARSE_CFG = 7.5 #@param {"type":"number"}
SLAT_STEPS = 12 #@param {"type":"integer"}
SLAT_CFG = 3.0 #@param {"type":"number"}
#@markdown TRELLIS.2: SLAT şekil için; doku adımı aynı/doku guidance1 sabit. 12–200 küresel görüş; render512–2048px; SH0–3; fitting100–30000; cap1000–3000000 üst sınır, adaptif büyüme. Max detay: 1536 cascade/4096 doku/160 görüş/1536px/1.5M cap/30k/SH2; en az60GiB, saatler sürebilir (ölçülmedi). Native TRELLIS kullanmaz.
TRELLIS2_PIPELINE_TYPE = '512' #@param ["512", "1024_cascade", "1536_cascade"]
MESH_RENDER_RESOLUTION = 1024 #@param {"type":"integer"}
MESH_SH_DEGREE = 2 #@param {"type":"integer"}
MESH_TEXTURE_SIZE = 2048 #@param [2048, 4096] {"type":"raw"}
MESH_VIEWS = 24 #@param {"type":"integer"}
MESH_FIT_ITERATIONS = 1500 #@param {"type":"integer"}
MESH_SPLAT_CAP = 50000 #@param {"type":"integer"}
#@markdown Çıktı üst sınırı: None orijinal sayıyı korur; sayı deterministik/kayıplı budama yapar, üretim VRAM'ini azaltmaz.
TARGET_SPLAT_COUNT = None #@param {"type":"raw"}
OUTPUT_DIR = '/content/drive/MyDrive/GaussianTests/text_to_splat/red-sports-car' #@param {"type":"string"}


## 2. Taşınabilir tarif ve GPU ön kontrolü

Kaynaklar notebook'a gömülür; Colab çekirdeği korunur. Ön kontrol gerçek VRAM'i, sürücüyü,
diski, CPU RAM'ini, erişimi ve mesh yolu için derleyiciyi kontrol eder.
GPU adındaki 80/96 GB tek başına yeterli değildir; uygun sürücü/derleyici gerekir.


In [ ]:
RUNTIME_FILES = {}
QUALITY_PRESETS = {}

import hashlib
import importlib
import json
import sys
from pathlib import Path

WORK = Path('/content/text-to-splat-v2')
if not RUNTIME_FILES or QUALITY_PRESET not in QUALITY_PRESETS:
    raise RuntimeError('Notebook Generator üzerinden notebook oluşturun; boş şablon doğrudan çalıştırılamaz.')
WORK.mkdir(parents=True, exist_ok=True)
for name, source in RUNTIME_FILES.items():
    if Path(name).name != name or not name.startswith('text_to_splat_'):
        raise RuntimeError('Gömülü tarif dosya adı geçersiz.')
    if name.endswith('.py'):
        compile(source, name, 'exec')
    (WORK / name).write_text(source, encoding='utf8')
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
import text_to_splat_bootstrap
bootstrap = importlib.reload(text_to_splat_bootstrap)
from text_to_splat_settings import GPU_PRESETS, validate_settings
CONFIG = {
    'prompt': PROMPT, 'negative_prompt': NEGATIVE_PROMPT, 'style': STYLE,
    'seed': SEED, 'quality_preset': QUALITY_PRESET, 'quality': QUALITY_PRESETS[QUALITY_PRESET],
    'output_dir': OUTPUT_DIR, 'target_splat_count': TARGET_SPLAT_COUNT, 'work_dir': str(WORK),
    'gpu_preset': GPU_PRESET, 'image_model': IMAGE_MODEL, 'background_model': BACKGROUND_MODEL,
    'reconstruction_model': RECONSTRUCTION_MODEL, 'image_steps': IMAGE_STEPS,
    'image_guidance': IMAGE_GUIDANCE, 'image_resolution': IMAGE_RESOLUTION,
    'trellis_seed': TRELLIS_SEED, 'sparse_steps': SPARSE_STEPS, 'sparse_cfg': SPARSE_CFG,
    'slat_steps': SLAT_STEPS, 'slat_cfg': SLAT_CFG, 'mesh_views': MESH_VIEWS,
    'trellis2_pipeline_type': TRELLIS2_PIPELINE_TYPE, 'mesh_render_resolution': MESH_RENDER_RESOLUTION,
    'mesh_sh_degree': MESH_SH_DEGREE, 'mesh_texture_size': MESH_TEXTURE_SIZE,
    'mesh_fit_iterations': MESH_FIT_ITERATIONS, 'mesh_splat_cap': MESH_SPLAT_CAP,
    'recipe_sha256': hashlib.sha256(json.dumps(RUNTIME_FILES, sort_keys=True).encode()).hexdigest(),
}
if type(APPLY_GPU_PRESET) is not bool:
    raise ValueError('APPLY_GPU_PRESET doğru/yanlış olmalı.')
if APPLY_GPU_PRESET:
    if GPU_PRESET not in GPU_PRESETS:
        raise ValueError('Geçersiz GPU_PRESET.')
    CONFIG.update(GPU_PRESETS[GPU_PRESET])
    CONFIG['trellis_seed'] = SEED
CONFIG = validate_settings(CONFIG)
bootstrap.host_preflight(CONFIG)
print('Etkin model/kalite ayarları:', {k: CONFIG[k] for k in GPU_PRESETS[GPU_PRESET]})


## 3. Google Drive bağlantısı

Colab'ın Drive yetkilendirme penceresini tamamlayın. Ara görüntü, Gaussian verisi, PLY, GIF,
manifest ve günlükler OUTPUT_DIR altında tutulur. Ayarlar uyuşmazsa yeni klasör seçmeniz istenir.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
bootstrap.validate_config(CONFIG)
print('Çıktı klasörü:', OUTPUT_DIR)


## 4. Birbirinden bağımsız özel ortamlar

Görüntü: Python3.11 / Torch2.7.1 cu128 / Diffusers0.37.0; native TRELLIS: Torch2.4.0 cu121,
xformers0.0.27.post2 ve spconv2.3.6 hazır wheel paketleri.
Deneysel TRELLIS.2: ayrı Torch2.7.1 cu128; CuMesh, FlexGEMM, nvdiffrast ve o-voxel
**bu özel ortamda derlenir**, gsplat1.5.3 CUDA kodu ilk kontrolde derlenebilir.
CUDA toolkit12.8/12.9 nvcc ve g++ yoksa model indirmeden Türkçe hata verir (CUDA13 bu cu128 tarifine uymaz).
Minik CUDA kontrolleri ve HF erişim kontrolü model ağırlıklarından önce çalışır.
Kernel yeniden başlatılmaz. İlk kurulum uzun sürebilir; ortam aynı oturumda yeniden kullanılır.


In [ ]:
bootstrap.prepare_environment(CONFIG)

## 5. Metin → referans görüntü

Seçilen model modern ortamda input.png üretir; seçilen U2Net/BiRefNet cutout.png üretir.
Bu adım bittiğinde GPU süreci kapanır; PNG geometri ortamına aktarılır.
Nesne eksikse/yanlışsa istemi değiştirip yeni OUTPUT_DIR kullanın.


In [ ]:
bootstrap.run_stage(CONFIG, 'image')
from IPython.display import display, Image
display(Image(filename=str(Path(OUTPUT_DIR) / 'input.png'), width=384))

## 6. Görüntü → Gaussian / deneysel mesh fitting

TRELLIS yalnız Gaussian decoder kullanır. TRELLIS.2 seçilen pipeline ile UV dokulu mesh.glb üretir,
seçilen çözünürlükte küresel görüşler çizer ve sınırlı DefaultStrategy büyümesiyle gsplat eğitir.
Mesh, görünüş ve fitting ayrı süreçlerdir. mesh.glb / mesh.npz / mesh_views.npz / mesh_images.npy / mesh_masks.npy / views_preview.jpg / gaussians_raw.npz
Drive'da SHA256 ile saklanır; tamamlanan aşamalardan devam edilir.
Görüşlerin her 12’ncisi PSNR/SSIM değerlendirmesine ayrılır; manifest tüm ayarları ve ölçümleri tutar. Beyaz arka planlı UV temel renk + alpha fitting yapılır; metal/roughness ve fiziksel PBR görünümü korunmaz.
SH derecesi her 1000 adımda artırılır; DefaultStrategy kapasiteyi aşmadan büyür. Büyük cap daha iyi sonuç garantisi değildir.


In [ ]:
bootstrap.run_stage(CONFIG, 'gaussian')

## 7. Standart PLY ve döner önizleme

Konumlar orijine merkezlenir; en uzun Gaussian merkez sınırı 1 birimdir. Sağ elli **+Y yukarı**:
TRELLIS (x,y,z) → (x,z,-y). Quaternionlar aynı rotasyonla dönüştürülür.
PLY: opacity logit, scale log, rot wxyz; gerçek SH0 renk, 45 sıfır f_rest alanı.
COLMAP dünya ekseni ve ölçeği keyfidir; bu nesne metre cinsinden değildir. Scene Editor'de döndürün/ölçekleyin.
GIF CPU'da en önemli 12.000 Gaussian ile yaklaşık ortografik SH0 önizlemedir; tam PLY daha ayrıntılı olabilir.


In [ ]:
bootstrap.run_stage(CONFIG, 'export')
display(Image(filename=str(Path(OUTPUT_DIR) / 'turntable.gif')))

## 8. Sonuçları uygulamada kullanın

Drive → OUTPUT_DIR → **target.ply** dosyasını bilgisayarınıza indirin.
Scene Editor'de PLY'yi varlık olarak içe aktarın ve sahneye ekleyin; orijindeki nesneyi mevcut sahnenize göre hizalayın.
Morph panelinde kaynak A olarak myroom, hedef B olarak bu varlığı seçin. Önce küçük örnek bütçesiyle
Particle/Shape seçeneklerini ve zaman sürgüsünü deneyin. Sonuç beğenilirse aynı B dosyasını tekrar kullanabilirsiniz.
manifest.json istemi, seed'i, model revizyonlarını, ortamı, aşama sürelerini, sayıları ve koordinat dönüşümünü kaydeder.

Hata/kesinti: Drive bağlıyken **Tümünü çalıştır** yeniden denenebilir. Aynı klasör farklı ayarları kabul etmez.
Bozuk/yarım aşama yeniden üretilir; günlükler OUTPUT_DIR/logs ve /content/text-to-splat-v2/logs altındadır.


In [ ]:
result = Path(OUTPUT_DIR)
for name in ('target.ply', 'turntable.gif', 'input.png', 'cutout.png', 'manifest.json'):
    print('✓', result / name)
print((result / 'manifest.json').read_text(encoding='utf8'))